In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Proj
from scipy.spatial import cKDTree
from tqdm.notebook import tqdm
import time
from datetime import datetime
from os.path import join
import multiprocessing
from functools import partial
from multiprocessing import Pool
from itertools import product

tqdm.pandas()

### Begin home location to MRMS matching

#### Load home location file

In [4]:
### TODO:change to new home location file generated without 1ping/day filtering once it has been generated

home_loc_fn = "/glade/derecho/scratch/ggantos/CAT/home_locations_tao_filtered_2024/home_loc_by_row_group.parquet"
home_loc_df = pd.read_parquet(home_loc_fn, engine='pyarrow')
print(home_loc_df.shape, home_loc_df.index.unique().shape)
home_loc_fn_23 = "/glade/derecho/scratch/ggantos/CAT/home_locations_tao_filtered_2023/home_loc_by_row_group.parquet"
home_loc_df_23 = pd.read_parquet(home_loc_fn_23, engine='pyarrow')
print(home_loc_df_23.shape, home_loc_df_23.index.unique().shape)
home_loc_df = home_loc_df[home_loc_df.index.isin(home_loc_df_23.index)]
print(home_loc_df.shape, home_loc_df.index.unique().shape)

(21117, 8) (21117,)
(16034, 8) (16034,)
(2684, 8) (2684,)


,duration_home,lat_home,lng_home,duration_work,lat_work,lng_work,zipcode,row_group
uid,,,,,,,,
001ce05b8f38b3584bc29e51d1ced1556b41bc3f9b4cb2e00dd2a0d3fa050979,7 days 19:24:20,35.581416,-119.354946,1 days 11:57:29,35.464193,-119.211174,93280,24
0039b8b63c8f142282ecd6e86ac40483bc52a0663bdd9ede828900572d3172bd,12 days 15:06:33,38.427812,-122.758841,3 days 07:16:53,38.496026,-122.768512,95407,49
004988492810559122f298d8da64b152d0545cf4195475358234eef096b05ab4,18 days 12:49:59,34.192521,-118.429261,10 days 21:23:09,34.228897,-118.529420,91401,62
0070d6b7bb343f88cabac6ca75ead7bf8e7dde6aeb8fbe23b796925ca613872a,40 days 22:20:11,37.730834,-122.426040,0 days 13:04:30,37.791191,-122.412827,94112,95
007ec691b48598010d03dd466d3033337e2f435a9ed9b82847592bb99f48e5be,14 days 13:17:34,37.017401,-121.461395,5 days 23:40:14,37.057852,-121.668833,95020,107
...,...,...,...,...,...,...,...,...
ffd3471682559103d752047cd9add71288654d39bdd60e055d91bf58560fc8f4,19 days 01:37:05,37.768053,-122.421382,11 days 12:08:10,37.821069,-122.286702,94103,55415
ffe3db49bc6a1b2a7eaea60138e3b1e8ed48d64399d93d8415420058afc9b906,9 days 23:47:47,34.054311,-117.759082,1 days 17:51:08,34.146297,-118.081650,91766,55429
fff261371b485aaa760cc1bddf7d72b1f13d7121f0616ac39197cb62db6e860f,20 days 15:43:45,34.089717,-118.172191,10 days 06:10:17,34.089814,-118.157898,90032,55441


In [ ]:
with open("../config/zips_counties.yaml", 'r') as stream:
    zips_counties = yaml.load(stream, Loader=yaml.FullLoader)
    zips = [str(z) for z in zips_counties["zips_LA"] + zips_counties["zips_SB"] + zips_counties["zips_VE"]]
home_loc_df = home_loc_df[home_loc_df.zipcode.isin(zips)]

zips_dict = {}
for k,v in zips_counties.items():
    for x in v:
        zips_dict[str(x)] = k.split("_")[1]
home_loc_df["county"] = home_loc_df["zipcode"].map(zips_dict)
home_loc_df

### Match home locations to nearest MRMS data

In [7]:
def create_tree(mrms_ds):
    # define projection
    proj = Proj(proj="lcc", R=6371229, lat_0=25, lon_0=265, lat_1=25, lat_2=25)
    
    # grid and project MRMS lat-lon coords then create KDTree
    mrms_lon_grid = np.tile(mrms_ds['lon'].values-360, (mrms_ds['lat'].values.shape[0], 1)).T
    mrms_lat_grid = np.tile(mrms_ds['lat'].values, (mrms_ds['lon'].values.shape[0], 1))
    proj_mrms_lon_grid, proj_mrms_lat_grid = np.array(proj(mrms_lon_grid, mrms_lat_grid))
    tree = cKDTree(np.c_[proj_mrms_lon_grid.ravel(), proj_mrms_lat_grid.ravel()])
    return tree, proj, mrms_lon_grid.shape

def get_mrms_var(df_sub, idx, mrms_ds, variable, variable_name):

    # grab MRMS data using calculated index
    lon_idx_da = xr.DataArray(idx[:, 0], dims="lon")
    lat_idx_da = xr.DataArray(idx[:, 1], dims="lat")
    mrms_idx = mrms_ds.isel(lon=lon_idx_da).isel(lat=lat_idx_da)[variable].values
    mrms_idx = np.swapaxes(mrms_idx, 0, 2)
    mrms_idx = mrms_idx[np.arange(0, mrms_idx.shape[0]), np.arange(0, mrms_idx.shape[1]), :]
    series = pd.Series(mrms_idx.tolist(), name=variable_name, index=df_sub.index)
    return series

def cycle_through_mrms_variables(row_group, home_loc_df, variables, date_strs, variable_aggregates):

    mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variables[0]}_{date_strs[0]}_daily_{variable_aggregates[0]}.nc"
    mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
    tree, proj, mrms_lon_grid_shape = create_tree(mrms_ds)
    
    df_sub = home_loc_df[home_loc_df.row_group == row_group]
    _, points = tree.query(np.column_stack(proj(df_sub['lng_home'], df_sub['lat_home'])))
    idx = np.column_stack(np.unravel_index(points, mrms_lon_grid_shape))

    for variable, date_str, variable_aggregate in product(variables, date_strs, variable_aggregates):
        mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variable}_{date_str}_daily_{variable_aggregate}.nc"
        mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
        variable_name = f"streamflow_{variable_aggregate}" if "STREAMFLOW" in variable else f"precip_{variable_aggregate}"
        series = get_mrms_var(df_sub, idx, mrms_ds, variable, variable_name)
        if variable_name in df_sub.columns:
            df_sub[variable_name] = df_sub[variable_name] + series
        else:
            df_sub = df_sub.merge(series, left_index=True, right_index=True)
    return df_sub

In [8]:
date_strs = ["2024-03-15T00:00_2024-04-29T00:00", "2023-03-15T00:00_2023-04-29T00:00"]
variables = ["MultiSensor_QPE_01H_Pass2_00.00", "FLASH_HP_MAXUNITSTREAMFLOW_00.00"]
variable_aggregates = ["max", "sum"]
row_groups = home_loc_df.row_group.unique()

n_processors = multiprocessing.cpu_count()
f = partial(cycle_through_mrms_variables,
            home_loc_df=home_loc_df,
            variables=variables,
            date_strs=date_strs,
            variable_aggregates=variable_aggregates)

s = time.time()
print('Beginning data processing...')
with Pool(processes=n_processors) as pool:
    result = pool.map_async(f, row_groups)
    results = result.get()
print(f"{np.round((time.time() - s) / 60, 2)} minutes to complete parallel computation.")
combined_data = pd.concat(results)
combined_data

Beginning data processing...


/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/pyproj/transformer.py:817: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return self._transformer._transform_point(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/pyproj/transformer.py:817: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return self._transformer._transform_point(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/pyproj/transformer.py:817: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return self._transformer._transform_point(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/pyproj/transformer.py:817: FutureWarning: Calling float on a single element Series is deprecated an

2.04 minutes to complete parallel computation.


,duration_home,lat_home,lng_home,duration_work,lat_work,lng_work,zipcode,row_group,precip_max,precip_sum,streamflow_max,streamflow_sum
uid,,,,,,,,,,,,
001ce05b8f38b3584bc29e51d1ced1556b41bc3f9b4cb2e00dd2a0d3fa050979,7 days 19:24:20,35.581416,-119.354946,1 days 11:57:29,35.464193,-119.211174,93280,24,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.3, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.099..."
0039b8b63c8f142282ecd6e86ac40483bc52a0663bdd9ede828900572d3172bd,12 days 15:06:33,38.427812,-122.758841,3 days 07:16:53,38.496026,-122.768512,95407,49,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.2, 3.1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.7, 8.7, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.3, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 2.900..."
004988492810559122f298d8da64b152d0545cf4195475358234eef096b05ab4,18 days 12:49:59,34.192521,-118.429261,10 days 21:23:09,34.228897,-118.529420,91401,62,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.300...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 11.2,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3, ..."
0070d6b7bb343f88cabac6ca75ead7bf8e7dde6aeb8fbe23b796925ca613872a,40 days 22:20:11,37.730834,-122.426040,0 days 13:04:30,37.791191,-122.412827,94112,95,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.400...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 11.9,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.900..."
007ec691b48598010d03dd466d3033337e2f435a9ed9b82847592bb99f48e5be,14 days 13:17:34,37.017401,-121.461395,5 days 23:40:14,37.057852,-121.668833,95020,107,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.900...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 18.7,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.9, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
ffd3471682559103d752047cd9add71288654d39bdd60e055d91bf58560fc8f4,19 days 01:37:05,37.768053,-122.421382,11 days 12:08:10,37.821069,-122.286702,94103,55415,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.6, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 12.3,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.200..."
ffe3db49bc6a1b2a7eaea60138e3b1e8ed48d64399d93d8415420058afc9b906,9 days 23:47:47,34.054311,-117.759082,1 days 17:51:08,34.146297,-118.081650,91766,55429,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.700...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.200..."
fff261371b485aaa760cc1bddf7d72b1f13d7121f0616ac39197cb62db6e860f,20 days 15:43:45,34.089717,-118.172191,10 days 06:10:17,34.089814,-118.157898,90032,55441,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.900...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.6, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.9, ..."


In [ ]:
save_out_path = "/glade/campaign/cisl/aiml/cat_flood/mrms/home_locations_mrms/"
date_str = datetime.today().strftime("%Y%m%d_%H%M")
file_path = f"veraset_users_mrms_{date_str}.parquet"
combined_data.to_parquet(join(save_out_path, file_path))

### Calculate Mobility Metrics Baseline Statistics

In [5]:
all_dates = pd.date_range(date_strs[0].split("_")[0], date_strs[0].split("_")[1], freq="1d").union(pd.date_range(date_strs[1].split("_")[0], date_strs[1].split("_")[1], freq="1d"))
AR_dates = pd.date_range("2024-03-29", "2024-04-01", freq="1d")
baseline_dates = all_dates.difference(AR_dates)

In [11]:
column_heads=['base_all', 'base_weekend', 'base_weekday', 'base_fri', 'base_sat', 'base_sun', 'base_mon']
column_subs = ['mean', 'std', 'median', 'iqr', 'n']
columns = [c+'_'+cs for c in column_heads for cs in column_subs]
mobility_metrics = ['number_of_visits', 'distance_straight_line', 'radius_of_gyration', 'num_stops', 'max_distance_from_home']
columns = [mm+'_'+c for c in columns for mm in mobility_metrics]
column_heads_weekdays = {'base_all': [0,1,2,3,4,5,6],
                         'base_weekend': [5,6],
                         'base_weekday': [0,1,2,3,4],
                         'base_fri': [4],
                         'base_sat': [5],
                         'base_sun': [6],
                         'base_mon': [0]}

df = pd.DataFrame(index=combined_data.index, columns=columns)

series_list = []
for baseline_date in baseline_dates:
    weekday = baseline_date.day_of_week
    ### TODO: change to new analysis_ready_data directory generated without 1ping/day filtering once it has been run
    data_dir = f"/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20250805/{baseline_date.strftime('%Y')}/"
    fn = f"veraset_{baseline_date.strftime('%Y-%m-%d')}_PST_chunked.parquet"
    metrics = pd.read_parquet(data_dir + fn)
    print(f"Overlap UIDs: {np.intersect1d(metrics.uid.unique(), combined_data.index.unique()).shape}")
    metrics = metrics[metrics.uid.isin(combined_data.index)].set_index("uid", drop=True)

    for column_head in column_heads:
        if weekday in column_heads_weekdays[column_head]:
            for mobility_metric in mobility_metrics:
                col = mobility_metric + '_' + column_head
                series_list.append(pd.Series(metrics[mobility_metric], name=col, index=metrics.index))
        else:
            continue

Overlap UIDs: (2642,)
Overlap UIDs: (2657,)
Overlap UIDs: (2652,)
Overlap UIDs: (2649,)
Overlap UIDs: (2627,)
Overlap UIDs: (2641,)
Overlap UIDs: (2646,)
Overlap UIDs: (2651,)
Overlap UIDs: (2656,)
Overlap UIDs: (2655,)
Overlap UIDs: (2654,)
Overlap UIDs: (2641,)
Overlap UIDs: (2655,)
Overlap UIDs: (2650,)
Overlap UIDs: (2649,)
Overlap UIDs: (2647,)
Overlap UIDs: (2656,)
Overlap UIDs: (2653,)
Overlap UIDs: (2641,)
Overlap UIDs: (2651,)
Overlap UIDs: (2657,)
Overlap UIDs: (2655,)
Overlap UIDs: (2653,)
Overlap UIDs: (2660,)
Overlap UIDs: (2648,)
Overlap UIDs: (2637,)
Overlap UIDs: (2649,)
Overlap UIDs: (2661,)
Overlap UIDs: (2649,)
Overlap UIDs: (2560,)
Overlap UIDs: (2659,)
Overlap UIDs: (2650,)
Overlap UIDs: (2639,)
Overlap UIDs: (2653,)
Overlap UIDs: (2658,)
Overlap UIDs: (2653,)
Overlap UIDs: (2657,)
Overlap UIDs: (2659,)
Overlap UIDs: (2656,)
Overlap UIDs: (2644,)
Overlap UIDs: (2653,)
Overlap UIDs: (2652,)
Overlap UIDs: (2656,)
Overlap UIDs: (2657,)
Overlap UIDs: (2654,)
Overlap UI

In [29]:
def calculate_iqr(row):
    Q1 = row.quantile(0.25)
    Q3 = row.quantile(0.75)
    IQR = Q3 - Q1
    return IQR
    
series = pd.concat(series_list, axis=1, join="inner")
df = df[df.index.isin(series.index)]
for column_head in column_heads:
    for mobility_metric in mobility_metrics:
        col = mobility_metric + "_" + column_head
        df.loc[:, col + "_mean"] = series[col].mean(axis=1)
        df.loc[:, col + "_std"] = series[col].std(axis=1)
        df.loc[:, col + "_median"] = series[col].median(axis=1)
        df.loc[:, col + "_iqr"] = series[col].apply(calculate_iqr, axis=1)
        df.loc[:, col + "_n"] = series[col].count(axis=1)
df

,number_of_visits_base_all_mean,distance_straight_line_base_all_mean,radius_of_gyration_base_all_mean,num_stops_base_all_mean,max_distance_from_home_base_all_mean,number_of_visits_base_all_std,distance_straight_line_base_all_std,radius_of_gyration_base_all_std,num_stops_base_all_std,max_distance_from_home_base_all_std,...,number_of_visits_base_mon_iqr,distance_straight_line_base_mon_iqr,radius_of_gyration_base_mon_iqr,num_stops_base_mon_iqr,max_distance_from_home_base_mon_iqr,number_of_visits_base_mon_n,distance_straight_line_base_mon_n,radius_of_gyration_base_mon_n,num_stops_base_mon_n,max_distance_from_home_base_mon_n
uid,,,,,,,,,,,,,,,,,,,,,
0070d6b7bb343f88cabac6ca75ead7bf8e7dde6aeb8fbe23b796925ca613872a,97.659091,31.821477,2.76825,5.943182,8.528589,42.85766,42.926537,4.557314,2.870211,13.058366,...,36.0,10.727653,1.3304,3.0,2.580195,12,12,12,12,12
007ec691b48598010d03dd466d3033337e2f435a9ed9b82847592bb99f48e5be,485.284091,185.564899,12.674814,6.318182,37.96264,205.771883,98.346183,6.373162,2.103885,20.81569,...,264.5,54.247052,4.270936,3.25,19.13735,12,12,12,12,12
00c05dbbc0a3005ff1fdb83f21c5757bb00e0fdd03c769376f80ba325a6a4b7e,47.556818,201.744058,16.037622,6.215909,50.071949,32.005336,196.226961,12.027357,3.442296,44.710668,...,34.5,281.347677,8.247522,5.25,23.023121,12,12,12,12,12
00e0483402e51cd0cb72c699a3b0a49f1cb9dd0165c3fcb0754064628ed77cdd,529.5,176.117564,14.946526,8.784091,41.933402,249.38688,169.269778,22.439117,3.39861,66.262986,...,232.5,201.748404,3.047089,6.75,16.35852,12,12,12,12,12
0139c27600dfb71e0ac78705698841e6610eee05428e45e62fbda5eed04fdae4,171.522727,66.744709,6.063675,5.034091,16.551053,104.164655,44.060036,2.017705,2.18379,13.229291,...,95.25,31.619568,1.62324,4.25,0.387542,12,12,12,12,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ff39619779f97fb2dbce197c4d3d72d344c93b86f4a87b108ed717859a5ae244,616.318182,74.240666,2.150694,8.886364,6.971035,288.528592,49.966698,1.592765,2.301583,6.207924,...,155.0,90.930835,2.27141,2.5,7.57336,12,12,12,12,12
ff469ed13f84e0e0523141c6872d2d8a66b5bc690c1fa60d196eaba8df827c90,330.511364,103.084506,7.911884,5.522727,23.452638,212.151648,116.23386,10.699904,2.836451,33.173837,...,175.75,56.370985,1.668946,4.25,2.975733,12,12,12,12,12
ffd3471682559103d752047cd9add71288654d39bdd60e055d91bf58560fc8f4,484.125,105.822785,6.829979,6.113636,18.031148,156.254862,60.940119,3.46753,2.54356,12.932548,...,91.75,27.314442,1.106038,0.25,1.060926,12,12,12,12,12


In [10]:
all_data = combined_data.merge(df, left_index=True, right_index=True)
save_out_path = "/glade/campaign/cisl/aiml/cat_flood/mrms/home_locations_mrms_mobility_stats/"
date_str = datetime.today().strftime("%Y%m%d_%H%M")
file_path = f"veraset_users_mrms_{date_str}_stats.parquet"
all_data.to_parquet(join(save_out_path, file_path))

### Calculate Deltas between AR Mobility Metrics and Baseline Mobility Metrics Statistics

In [30]:
column_heads=['base_all', 'base_weekend', 'base_weekday', 'base_fri', 'base_sat', 'base_sun', 'base_mon']
mobility_metrics = ['number_of_visits', 'distance_straight_line', 'radius_of_gyration', 'num_stops', 'max_distance_from_home']
column_heads_weekdays = {'base_all': [0,1,2,3,4,5,6],
                         'base_weekend': [5,6],
                         'base_weekday': [0,1,2,3,4],
                         'base_fri': [4],
                         'base_sat': [5],
                         'base_sun': [6],
                         'base_mon': [0]}

series_list = []
for AR_date in AR_dates:
    weekday = AR_date.day_of_week
    ### TODO: change to new analysis_ready_data directory generated without 1ping/day filtering once it has been run
    data_dir = f"/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20250805/{AR_date.strftime('%Y')}/"
    fn = f"veraset_{AR_date.strftime('%Y-%m-%d')}_PST_chunked.parquet"
    metrics = pd.read_parquet(data_dir + fn)
    metrics = metrics[metrics.uid.isin(all_data.index)].set_index("uid", drop=True)
    for column_head in column_heads:
        if weekday in column_heads_weekdays[column_head]:
            for mobility_metric in mobility_metrics:
                col = mobility_metric + '_' + column_head
                try:
                    delta = (metrics[mobility_metric] - all_data[col+'_mean'])/all_data[col+'_std']
                except:
                    delta = (metrics[mobility_metric] - all_data[col+'_mean'])/all_data[col+'_std']
                series_list.append(pd.Series(delta, name=f"{AR_date.strftime('%Y-%m-%d')}_{col}_z", index=all_data.index))
                try:
                    delta = (metrics[mobility_metric] - all_data[col+'_median'])/all_data[col+'_iqr']
                except:
                    delta = (metrics[mobility_metric] - all_data[col+'_median'])/all_data[col+'_iqr']
                series_list.append(pd.Series(delta, name=f"{AR_date.strftime('%Y-%m-%d')}_{col}_modz", index=all_data.index))
        else:
            continue
df_deltas = pd.concat(series_list, axis=1, join="inner")
df_deltas

,2024-03-29_number_of_visits_base_all_z,2024-03-29_number_of_visits_base_all_modz,2024-03-29_distance_straight_line_base_all_z,2024-03-29_distance_straight_line_base_all_modz,2024-03-29_radius_of_gyration_base_all_z,2024-03-29_radius_of_gyration_base_all_modz,2024-03-29_num_stops_base_all_z,2024-03-29_num_stops_base_all_modz,2024-03-29_max_distance_from_home_base_all_z,2024-03-29_max_distance_from_home_base_all_modz,...,2024-04-01_number_of_visits_base_mon_z,2024-04-01_number_of_visits_base_mon_modz,2024-04-01_distance_straight_line_base_mon_z,2024-04-01_distance_straight_line_base_mon_modz,2024-04-01_radius_of_gyration_base_mon_z,2024-04-01_radius_of_gyration_base_mon_modz,2024-04-01_num_stops_base_mon_z,2024-04-01_num_stops_base_mon_modz,2024-04-01_max_distance_from_home_base_mon_z,2024-04-01_max_distance_from_home_base_mon_modz
uid,,,,,,,,,,,,,,,,,,,,,
0070d6b7bb343f88cabac6ca75ead7bf8e7dde6aeb8fbe23b796925ca613872a,-0.062045,0.038835,-0.261306,0.153844,-0.360294,-0.725275,0.368202,0.25,-0.251527,-0.001952,...,0.044227,0.291667,-0.392361,-0.027017,-0.714686,-0.571988,-1.044639,-0.666667,-0.895463,-1.026566
007ec691b48598010d03dd466d3033337e2f435a9ed9b82847592bb99f48e5be,1.602337,1.324723,1.147399,1.115135,0.252235,0.404378,0.324076,0.333333,0.179344,0.154946,...,-1.145362,-0.604915,-0.274559,-0.590194,0.075746,0.369865,-0.205174,-0.307692,0.562011,0.291149
00c05dbbc0a3005ff1fdb83f21c5757bb00e0fdd03c769376f80ba325a6a4b7e,1.85729,2.014706,1.581481,1.480017,0.28636,0.488415,3.423323,2.4,0.000402,0.250349,...,-0.450936,-0.318841,1.848738,1.730315,2.842627,5.111732,0.214732,0.47619,2.333637,7.138246
00e0483402e51cd0cb72c699a3b0a49f1cb9dd0165c3fcb0754064628ed77cdd,-1.974843,-1.827893,-0.593226,-0.343471,-0.235168,0.264639,-1.701899,-1.2,-0.267811,0.034904,...,0.101409,0.365591,8.33979,4.840715,14.108156,46.175574,-0.715248,-0.296296,16.865293,26.19414
0139c27600dfb71e0ac78705698841e6610eee05428e45e62fbda5eed04fdae4,-1.166641,-0.829659,-0.773499,-0.669235,-0.36665,-0.475768,-1.389369,-1.0,-0.360262,-2.998461,...,-0.991122,-0.976378,-0.100351,0.226528,-0.021993,0.353479,-0.133414,0.0,-0.253631,0.709444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ff39619779f97fb2dbce197c4d3d72d344c93b86f4a87b108ed717859a5ae244,-0.247179,-0.155311,0.561752,0.825222,0.756214,0.919601,0.483857,0.444444,0.964329,1.667602,...,-0.677452,-1.112903,-0.061106,0.072477,0.855182,0.696803,0.384512,0.2,0.865713,0.620701
ff469ed13f84e0e0523141c6872d2d8a66b5bc690c1fa60d196eaba8df827c90,-0.327649,-0.224422,-0.062912,0.3617,-0.403387,-0.137786,0.520817,0.4,-0.442965,-0.187812,...,-0.943968,-0.594595,-0.807235,-0.650683,-0.39203,-0.3813,-1.018138,-0.470588,-0.352572,-0.747827
ffd3471682559103d752047cd9add71288654d39bdd60e055d91bf58560fc8f4,-0.051998,0.122156,0.185766,0.481548,-0.116299,0.091133,0.348474,0.25,-0.284057,0.957188,...,-0.422773,-0.163488,0.369631,0.584588,1.196404,0.964853,-0.149525,0.0,1.3606,1.00397


In [35]:
all_data = all_data.merge(df_deltas, left_index=True, right_index=True)
save_out_path = "/glade/campaign/cisl/aiml/cat_flood/mrms/home_locations_mrms_mobility_stats_deltas/"
date_str = datetime.today().strftime("%Y%m%d_%H%M")
file_path = f"veraset_users_mrms_{date_str}_stats_deltas.parquet"
all_data.to_parquet(join(save_out_path, file_path))
all_data